In [3]:
from typing import TypedDict

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver # In memory checkpointing for graph -> Short term memory
from langgraph.store.memory import InMemoryStore # Long term memory store for graph -> Long term memory

In [5]:
class AgentState(MessagesState):
  intent: str
  order_id: str
  preferred_language: str

In [6]:
class Context(TypedDict):
  user_id: str
  plan: str

In [7]:
def parse_request(state: AgentState, config: RunnableConfig) -> dict:
  user_text= str(state['messages'][-1].content).strip()
  thread_id= config['configurable']['thread_id']

  print(f"[Node1] thread_id: {thread_id}")
  print(f"[Node1] message_count: {len(state['messages'])}")
  print(f"[Node1] existing_order: {state['order_id']}")

  if user_text.startswith("My order_id is"):
    order_id= user_text.removeprefix("My order_id is ").strip()

    return {
      "order_id": order_id,
      "intent": "save_order"
    }
  
  if user_text == "Where is my order":
    return {
      "intent": "track_order"
    }
  
  if user_text.startswith("Remember my language "):
    return { "intent": "save_language" }
  
  if user_text == "Which language do I prefer?":
    return {"intent": "ask_language"}
  
  return {"intent": "unknown"}
  

In [9]:
def load_user_memory(
    state: AgentState,
    runtime: Runtime[Context]
) -> dict:
  user_id = runtime.context['user_id']
  plan = runtime.context['plan']

  # Load user memory from the long-term store
  store = runtime.store

  namespace = ("users", user_id)

  if state['intent'] == "save_language":
    message = str(state['messages'][-1].content).strip()
    language = message.removeprefix("Remember my language ").strip()

    store.put(
      namespace=namespace,
      key="preferred_language",
      value={"value": language}
    )

  memory = store.get(namespace, "preferred_language")
  language = (
    memory.value['value']
    if memory is not None else "not set"
  )

  return {
    "preferred_language": language
  }


In [10]:
def respond(state: AgentState, config: RunnableConfig, runtime: Runtime[Context]) -> dict:
  thread_id= config['configurable']['thread_id']
  user_id = runtime.context['user_id']
  intent = state['intent']

  if intent == "save_order":
    answer = {
      "response": f"Order ID {order_id} saved for user {user_id} in thread {thread_id}."
    }
  elif intent == "track_order":
    
    if state.get("order_id"):
      answer = {
        "response": f"Tracking order {order_id} for user {user_id} in thread {thread_id}."
      }
    elif intent == "save_language":
      answer = {
        "response": f"Preferred language '{language}' saved for user {user_id}."
      }
    
    elif intent == "ask_language":
      if state.get('preferred_language'):
        answer = {
        "response": f"User {user_id}'s preferred language is '{preferred_language}'."
      }    
    else:
      answer = {
        "response": f"No order ID found for user {user_id}. Please provide your order ID."
      }

  return {
    "messages": [AIMessage(content=answer['response'])]
  }



In [11]:
builder = StateGraph(AgentState , context_schema=Context)

builder.add_node("parse_request", parse_request)
builder.add_node("load_user_memory", load_user_memory)
builder.add_node("respond", respond)

builder.add_edge(START, "parse_request")
builder.add_edge("parse_request", "load_user_memory")
builder.add_edge("load_user_memory", "respond")
builder.add_edge("respond", END)

checkpointer = InMemorySaver() # In memory checkpointing for graph -> Short term memory
store = InMemoryStore()  # Long term memory store for graph -> Long term memory

graph = builder.compile(
  checkpointer=checkpointer,
  store=store
)